<a href="https://colab.research.google.com/github/Japhet4/App-Website/blob/main/LuxTTS_Complete_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ LuxTTS - High-Quality Voice Cloning TTS

**Features:**
- 🎭 Voice cloning from audio samples
- 🚀 150x realtime speed
- 🎵 48kHz high-quality audio
- 📱 Works on GPU and CPU
- 🎨 Clone any voice with just 3-10 seconds of audio

**Note:** Make sure to enable GPU runtime for better performance:
- Runtime → Change runtime type → Hardware accelerator → GPU (T4)

## Step 1: Clone Repository and Install Dependencies

In [1]:
# Clone LuxTTS repository
!git clone https://github.com/ysharma3501/LuxTTS.git
%cd LuxTTS
!pip install -r requirements.txt
%cd ..

# Install additional dependencies
!pip install gradio -q

Cloning into 'LuxTTS'...
remote: Enumerating objects: 123, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 123 (delta 54), reused 18 (delta 18), pack-reused 33 (from 1)
Receiving objects: 100% (123/123), 166.67 KiB | 1.28 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/LuxTTS
Looking in links: https://k2-fsa.github.io/icefall/piper_phonemize.html
  Cloning https://github.com/ysharma3501/LinaCodec.git to /tmp/pip-req-build-nnn2f1wr
  Running command git clone --filter=blob:none --quiet https://github.com/ysharma3501/LinaCodec.git /tmp/pip-req-build-nnn2f1wr
  Resolved https://github.com/ysharma3501/LinaCodec.git to commit c0ae7c7285e121475c27592cfbb600624b714290
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Step 2: Import Libraries and Setup

In [2]:
# Add LuxTTS to Python path
import sys
sys.path.append('/content/LuxTTS')

import torch
import gradio as gr
import numpy as np
import soundfile as sf
from IPython.display import Audio
import warnings
import os
warnings.filterwarnings('ignore')

from zipvoice.luxvoice import LuxTTS

print("✓ Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

✓ Libraries imported successfully!
PyTorch version: 2.10.0+cu128
CUDA available: True


## Step 3: Load LuxTTS Model

In [3]:
print("Loading LuxTTS model...")

# Detect device
if torch.cuda.is_available():
    device = 'cuda'
    print("🚀 Using GPU")
else:
    device = 'cpu'
    print("💻 Using CPU (slower but will work)")

# Load model
lux_tts = LuxTTS('YatharthS/LuxTTS', device=device, threads=2)
print("✓ LuxTTS model loaded successfully!")

Loading LuxTTS model...
🚀 Using GPU


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/697 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Device set to use cuda


Loading model on GPU
✓ LuxTTS model loaded successfully!


## Step 4: Define Voice Cloning Function

In [4]:
def clone_voice_and_generate(text, reference_audio, rms=0.01, t_shift=0.9,
                             num_steps=4, speed=1.0, return_smooth=False,
                             ref_duration=5):
    """
    Generate speech with voice cloning from reference audio

    Args:
        text: Input text to convert to speech
        reference_audio: Path to reference audio file for voice cloning
        rms: Volume level (0.01 recommended)
        t_shift: Sampling parameter for quality (0.9 default)
        num_steps: Quality steps (3-4 recommended for speed/quality balance)
        speed: Speech speed (1.0 = normal)
        return_smooth: Make audio smoother (may reduce clarity)
        ref_duration: Reference audio duration to use (seconds)

    Returns:
        Audio file path and status message
    """
    try:
        if not text or text.strip() == "":
            return None, "⚠️ Please enter some text!"

        if reference_audio is None:
            return None, "⚠️ Please upload a reference audio file!"

        print("🎙️ Encoding reference audio...")

        # Encode the reference audio
        encoded_prompt = lux_tts.encode_prompt(
            reference_audio,
            duration=ref_duration,
            rms=rms
        )

        print("🎵 Generating speech...")

        # Generate speech
        final_wav = lux_tts.generate_speech(
            text,
            encoded_prompt,
            num_steps=num_steps,
            t_shift=t_shift,
            speed=speed,
            return_smooth=return_smooth
        )

        # Convert to numpy array
        final_wav = final_wav.numpy().squeeze()

        # Save to file
        output_path = "/tmp/luxttts_output.wav"
        sf.write(output_path, final_wav, 48000)

        success_msg = f"✅ Speech generated successfully! ({len(text)} characters, 48kHz)"
        print(success_msg)
        return output_path, success_msg

    except Exception as e:
        error_msg = f"❌ Error: {str(e)}"
        print(error_msg)
        return None, error_msg

## Step 5: Create Gradio Interface

In [5]:
# Create the Gradio interface
with gr.Blocks(theme=gr.themes.Soft(), title="LuxTTS Voice Cloning") as demo:
    gr.Markdown(
        """
        # 🎙️ LuxTTS - High-Quality Voice Cloning

        Clone any voice and generate natural speech at 150x realtime speed!

        **How to use:**
        1. Upload a reference audio file (3-10 seconds of clear speech works best)
        2. Enter the text you want to generate
        3. Adjust parameters if needed
        4. Click Generate Speech
        """
    )

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 📝 Input")

            text_input = gr.Textbox(
                label="Text to Speak",
                placeholder="Enter the text you want to convert to speech...",
                lines=5
            )

            reference_audio = gr.Audio(
                label="Reference Voice Audio",
                type="filepath",
                sources=["upload", "microphone"]
            )

            gr.Markdown("### ⚙️ Advanced Parameters")

            with gr.Row():
                speed_slider = gr.Slider(
                    minimum=0.5,
                    maximum=2.0,
                    value=1.0,
                    step=0.1,
                    label="Speech Speed"
                )

                rms_slider = gr.Slider(
                    minimum=0.005,
                    maximum=0.05,
                    value=0.01,
                    step=0.005,
                    label="Volume (RMS)"
                )

            with gr.Row():
                num_steps = gr.Slider(
                    minimum=1,
                    maximum=10,
                    value=4,
                    step=1,
                    label="Quality Steps (3-4 recommended)"
                )

                t_shift = gr.Slider(
                    minimum=0.5,
                    maximum=1.0,
                    value=0.9,
                    step=0.05,
                    label="T-Shift (quality parameter)"
                )

            with gr.Row():
                ref_duration = gr.Slider(
                    minimum=1,
                    maximum=10,
                    value=5,
                    step=1,
                    label="Reference Duration (seconds)"
                )

                return_smooth = gr.Checkbox(
                    label="Smooth Output",
                    value=False
                )

            generate_btn = gr.Button("🎵 Generate Speech", variant="primary", size="lg")

        with gr.Column():
            gr.Markdown("### 🔊 Output")

            output_audio = gr.Audio(
                label="Generated Speech",
                type="filepath"
            )

            status_text = gr.Textbox(
                label="Status",
                interactive=False
            )

    gr.Markdown("### 📚 Example Texts")
    gr.Examples(
        examples=[
            ["Hello! Welcome to LuxTTS, the high-quality voice cloning system."],
            ["The quick brown fox jumps over the lazy dog."],
            ["This is an example of advanced text to speech with voice cloning capabilities."],
            ["LuxTTS can clone any voice from just a few seconds of audio."],
            ["Artificial intelligence is transforming how we interact with technology."],
            ["Hey, what's up? I'm feeling really great if you ask me honestly!"]
        ],
        inputs=text_input
    )

    gr.Markdown(
        """
        ### 💡 Tips for Best Results

        - **Reference Audio:** Use 3-10 seconds of clear speech without background noise
        - **Quality Steps:** 3-4 steps give the best speed/quality balance
        - **Speed:** 1.0 is normal, try 0.8 for slower or 1.2 for faster
        - **Volume (RMS):** 0.01 is recommended, increase for louder output
        - **Smooth Output:** Makes speech smoother but may reduce clarity
        - **T-Shift:** Higher values (0.9) give better quality but may have pronunciation errors
        """
    )

    generate_btn.click(
        fn=clone_voice_and_generate,
        inputs=[
            text_input,
            reference_audio,
            rms_slider,
            t_shift,
            num_steps,
            speed_slider,
            return_smooth,
            ref_duration
        ],
        outputs=[output_audio, status_text]
    )

print("✓ Gradio interface created!")

✓ Gradio interface created!


## Step 6: Launch the Interface

In [ ]:
# Launch the Gradio interface
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ee0f8dd27d2486d512.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 🧪 Additional Testing (Optional)

In [ ]:
# Test the TTS function directly with your own audio file
# Upload an audio file first, then run this cell

# Example:
# test_audio = '/content/your_audio.wav'
# test_text = "This is a test of the voice cloning system."
# audio_path, status = clone_voice_and_generate(test_text, test_audio)
# print(status)
# if audio_path:
#     from IPython.display import Audio, display
#     display(Audio(audio_path, rate=48000))

## 📝 Notes & Documentation

### Voice Cloning Tips:
1. **Reference Audio Quality Matters:** Use clear audio with minimal background noise
2. **Duration:** 3-10 seconds is ideal for reference audio (minimum 3 seconds)
3. **Consistency:** The reference voice should speak naturally and clearly
4. **Formats:** Supports WAV, MP3, and most common audio formats
5. **Multiple voices:** You can clone different voices by uploading different reference audio files

### Performance:
- **GPU (T4):** Extremely fast (150x realtime) - recommended
- **CPU:** Still usable but slower (real-time to 10x realtime)
- **VRAM Usage:** Fits within 1GB VRAM

### Parameters Explained:
- **Quality Steps (num_steps):** Higher = better quality but slower. 3-4 is the sweet spot for efficiency
- **T-Shift:** Controls pronunciation quality. 0.9 is recommended. Lower = fewer pronunciation errors but worse quality
- **Speed:** Adjusts how fast the speech is. 1.0 = normal, <1.0 = slower, >1.0 = faster
- **RMS (Volume):** Controls output volume. 0.01 is recommended for normal loudness
- **Smooth Output:** Can make speech smoother but may reduce clarity. Use if you hear metallic sounds
- **Reference Duration:** How many seconds of reference audio to use for cloning

### Common Issues & Solutions:
- **Metallic sounds:** Try enabling "Smooth Output" checkbox
- **Pronunciation errors:** Lower the T-Shift value (try 0.7-0.8)
- **Too quiet:** Increase RMS value (try 0.02-0.03)
- **Too slow generation:** Reduce quality steps to 3
- **Poor voice match:** Use a longer reference audio (5-10 seconds)

### Technical Details:
- **Output Quality:** 48kHz high-quality audio (better than most TTS at 24kHz)
- **Architecture:** Based on ZipVoice, distilled to 4 steps with improved sampling
- **Model Size:** Lightweight, fits in 1GB VRAM
- **License:** Apache-2.0

### Additional Resources:
- [GitHub Repository](https://github.com/ysharma3501/LuxTTS)
- [Hugging Face Model](https://huggingface.co/YatharthS/LuxTTS)
- [Hugging Face Demo Space](https://huggingface.co/spaces/YatharthS/LuxTTS)

---

**Enjoy using LuxTTS! 🎉**

If you find this useful, consider starring the [GitHub repo](https://github.com/ysharma3501/LuxTTS)!